# Before beginning, we would like to inform you that we decided to develop two methods from scratch.

# Parse data
Since each example consists of 4 lines, we process 4 lines at a time. We preprocess the sentences, record the entity tag positions, and remove the tags. Then we extract the relationship information and add the data to the list.

For example, the train file.txt looks like below

_10	"The solute was placed inside a beaker and 5 mL of the <e1>solvent</e1> was pipetted into a 25 mL glass <e2>flask</e2> for each trial."_

_Entity-Destination(e1,e2)_

_Comment:_

_(empty)_




In [1]:
import re
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import random_split
import random
np.random.seed(42)


def parse_data(file_path, is_test=False):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    if is_test:
        for line in lines:
            parts = line.split('\t')
            sentence = parts[1].strip()
            sentence = sentence.replace('< e1 >', '<e1>')
            sentence = sentence.replace('< e2 >', '<e2>')
            sentence = sentence.replace('< /e1 >', '</e1>')
            sentence = sentence.replace('< /e2 >', '</e2>')

            sentence = re.sub(' {2,}', ' ', sentence)
            e1_start = sentence.find('<e1>')
            e1_end = sentence.find('</e1>')
            e2_start = sentence.find('<e2>')
            e2_end = sentence.find('</e2>')

            sentence = sentence.replace('<e1>', '').replace('</e1>', '').replace('<e2>', '').replace('</e2>', '')

            data.append({
                'sentence': sentence.strip('\"'),
                'e1_start': e1_start,
                'e1_end': e1_end - 4,
                'e2_start': e2_start - 4 if e2_start > e1_end else e2_start - 8,
                'e2_end': e2_end - 4 if e2_end > e1_end else e2_end - 8,
                'relation': None
            })
    else:
        for i in range(0, len(lines), 4):  # Each example spans 4 lines
            sentence = lines[i].split('\t')[1].strip()
            sentence = sentence.replace('< e1 >', '<e1>')
            sentence = sentence.replace('< e2 >', '<e2>')
            sentence = sentence.replace('< /e1 >', '</e1>')
            sentence = sentence.replace('< /e2 >', '</e2>')

            sentence = re.sub(' {2,}', ' ', sentence)
            e1_start = sentence.find('<e1>')
            e1_end = sentence.find('</e1>')
            e2_start = sentence.find('<e2>')
            e2_end = sentence.find('</e2>')

            sentence = sentence.replace('<e1>', '').replace('</e1>', '').replace('<e2>', '').replace('</e2>', '')

            relation = lines[i + 1].strip()

            data.append({
                'sentence': sentence.strip('\"'),
                'e1_start': e1_start,
                'e1_end': e1_end - 4,
                'e2_start': e2_start - 4 if e2_start > e1_end else e2_start - 8,
                'e2_end': e2_end - 4 if e2_end > e1_end else e2_end - 8,
                'relation': relation
            })

    return data

The parse_data function is used to read training and validation data, extract **relationships (labels)** for each data, create a unique list of relationships (labels), and map them to an index.

The train_data looks like below

_{'sentence': 'The system as described above has its greatest application in an arrayed configuration of antenna elements.', 'e1_start': 74, 'e1_end': 87, 'e2_start': 104, 'e2_end': 116, 'relation': 'Component-Whole(e2,e1)'}_

In [2]:
train_data = parse_data('TRAIN_FILE.TXT')
valid_data = parse_data('TEST_FILE_FULL.TXT')
all_labels = set([item['relation'] for item in train_data] + [item['relation'] for item in valid_data])
label_map = {label: idx for idx, label in enumerate(sorted(all_labels))}

for sample in train_data[:5]:
    print(sample)

{'sentence': 'The system as described above has its greatest application in an arrayed configuration of antenna elements.', 'e1_start': 74, 'e1_end': 87, 'e2_start': 104, 'e2_end': 116, 'relation': 'Component-Whole(e2,e1)'}
{'sentence': 'The child was carefully wrapped and bound into the cradle by means of a cord.', 'e1_start': 5, 'e1_end': 10, 'e2_start': 57, 'e2_end': 67, 'relation': 'Other'}
{'sentence': 'The author of a keygen uses a disassembler to look at the raw assembly code.', 'e1_start': 5, 'e1_end': 11, 'e2_start': 36, 'e2_end': 52, 'relation': 'Instrument-Agency(e2,e1)'}
{'sentence': 'A misty ridge uprises from the surge.', 'e1_start': 9, 'e1_end': 14, 'e2_start': 37, 'e2_end': 46, 'relation': 'Other'}
{'sentence': 'The student association is the voice of the undergraduate student population of the State University of New York at Buffalo.', 'e1_start': 5, 'e1_end': 12, 'e2_start': 18, 'e2_end': 33, 'relation': 'Member-Collection(e1,e2)'}


The label map presented below.

In [3]:
label_map

{'Cause-Effect(e1,e2)': 0,
 'Cause-Effect(e2,e1)': 1,
 'Component-Whole(e1,e2)': 2,
 'Component-Whole(e2,e1)': 3,
 'Content-Container(e1,e2)': 4,
 'Content-Container(e2,e1)': 5,
 'Entity-Destination(e1,e2)': 6,
 'Entity-Destination(e2,e1)': 7,
 'Entity-Origin(e1,e2)': 8,
 'Entity-Origin(e2,e1)': 9,
 'Instrument-Agency(e1,e2)': 10,
 'Instrument-Agency(e2,e1)': 11,
 'Member-Collection(e1,e2)': 12,
 'Member-Collection(e2,e1)': 13,
 'Message-Topic(e1,e2)': 14,
 'Message-Topic(e2,e1)': 15,
 'Other': 16,
 'Product-Producer(e1,e2)': 17,
 'Product-Producer(e2,e1)': 18}

# Preprocess data
The preprocess_data function preprocesses the given data and converts it into a format that can be input to the model.

In [4]:
def preprocess_data(data, tokenizer, label_map):
    processed_data = []

    for item in data:
        sentence = item['sentence']

        e1_start, e1_end = item['e1_start'], item['e1_end']
        e2_start, e2_end = item['e2_start'], item['e2_end']


        sentence = (
            sentence[:e1_start] + '[E1]' + sentence[e1_start:e1_end] + '[/E1]' +
            sentence[e1_end:e2_start] + '[E2]' + sentence[e2_start:e2_end] + '[/E2]' +
            sentence[e2_end:]
        )

        # Tokenize the sentence
        inputs = tokenizer(sentence, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

        # Convert label to ID
        label = label_map[item['relation']]

        processed_data.append({
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'label': label
        })

    return processed_data

Below is the process of preprocessing text data and creating a PyTorch dataset and data loader to prepare it for model training.

Since there is no validation dataset in SemEval2010 task8, the train dataset was split 80:20 for training and testing.

In [5]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
#train_data = parse_data('TRAIN_FILE.TXT')
train_processed = preprocess_data(train_data, tokenizer, label_map)
input_ids = torch.stack([item['input_ids'] for item in train_processed])
attention_masks = torch.stack([item['attention_mask'] for item in train_processed])
labels_train = torch.tensor([item['label'] for item in train_processed])
train_dataset = TensorDataset(input_ids, attention_masks, labels_train)


train_size = int(0.8 * len(train_dataset))  # 80% for training
valid_size = len(train_dataset) - train_size  # Remaining 20% for validation
train_data, valid_data = random_split(train_dataset, [train_size, valid_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(valid_data, batch_size=32, shuffle=False)

valid_data = parse_data('TEST_FILE_FULL.TXT')
test_processed = preprocess_data(valid_data, tokenizer, label_map)
input_ids = torch.stack([item['input_ids'] for item in test_processed])
attention_masks = torch.stack([item['attention_mask'] for item in test_processed])
labels_test = torch.tensor([item['label'] for item in test_processed])
valid_dataset = TensorDataset(input_ids, attention_masks, labels_test)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)  # Don't shuffle test data

for sample in train_processed[:5]:
    print(sample)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': tensor([  101,  1996,  2291,  2004,  2649,  2682,  2038,  2049,  4602,  4646,
         1999,  2019,  9140,  2098,  1039,  1031,  1041,  2487,  1033,  2006,
         8873, 27390,  3370,  1031,  1013,  1041,  2487,  1033,  1997, 13438,
         3449, 21382,  2078,  1031,  1041,  2475,  1033, 24529,  1012,  1031,
         1013,  1041,  2475,  1033,   102,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

As the basic BiLSTM didn't make F1 score we expected, we  implemented a function that loads **GloVe (Global Vectors for Word Representation) embeddings** and stores words and their embedding vectors in the form of a dictionary.

In [29]:
def load_glove_embeddings(glove_path, embedding_dim):
    glove_dict = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.strip().split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector
    return glove_dict

# Load 300-dimensional embeddings
glove_path = "glove.6B.300d.txt"
glove_dict = load_glove_embeddings(glove_path, embedding_dim=300)

Below is a function that generates an embedding matrix using a tokenizer and GloVe embeddings.


In [30]:
def create_embedding_matrix(tokenizer, glove_dict, embedding_dim):
    vocab_size = len(tokenizer)
    embedding_matrix = np.random.uniform(-0.05, 0.05, (vocab_size, embedding_dim))  # Random initialization

    for word, idx in tokenizer.vocab.items():
        if word in glove_dict:
            embedding_matrix[idx] = glove_dict[word]  # Use pre-trained embedding

    return torch.tensor(embedding_matrix, dtype=torch.float32)

# Create the embedding matrix
embedding_matrix = create_embedding_matrix(tokenizer, glove_dict, embedding_dim=300)

# Define Attention based BiLSTM

In addition, we decide to add **attention mechanism** to make better F1 score





In [31]:
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix, hidden_dim, num_classes, dropout_rate=0.3):
        super(BiLSTMAttention, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout_lstm = nn.Dropout(dropout_rate)
        self.attention = nn.Linear(hidden_dim*2, 1)
        self.fc = nn.Linear(hidden_dim*2, num_classes)
        self.dropout_fc = nn.Dropout(dropout_rate)
    def forward(self, input_ids, attention_mask=None, labels=None):
        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = self.dropout_lstm(lstm_out)

        attention_scores = self.attention(lstm_out).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1)
        attention_output = torch.sum(lstm_out * attention_weights.unsqueeze(-1), dim=1)
        attention_output = self.dropout_fc(attention_output)

        logits = self.fc(attention_output)
        loss = None
        if labels != None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {'logits': logits, 'loss': loss}

# Training

epoch :10

metrics: Accuracy and F1

If the loss value for the first epoch is smaller than -ln(1/#class), it is a positive sign that the model is performing better than random guessing from the beginning.

SemEval2010 task8 has 19 classes.
-ln(1/19) = 2.94

Our loss for epoch is 2.27

2.27<2.94

In [32]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BiLSTMAttention(vocab_size=len(tokenizer), embedding_dim=300, embedding_matrix=embedding_matrix,hidden_dim=256, num_classes=len(label_map), dropout_rate=0.2).to(device)
optimizer = AdamW(model.parameters(), lr=1e-3)
acc = []
losses = []
max_acc = float('-inf')

for epoch in range(10):  # Number of epochs
    model.train()
    epoch_loss = 0
    name='custom_lstm'
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs['loss']
        loss.backward()
        epoch_loss += loss.item()
        optimizer.step()
    losses.append(epoch_loss/len(train_loader))
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in valid_loader:
            input_ids, attention_mask, labels = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs['logits']

            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    if accuracy > max_acc:
      max_acc = accuracy
      torch.save(model.state_dict(), f'best_model_{name}.pth')
    acc.append(accuracy)
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    mask = true_labels != label_map['Other']
    filtered_predictions = predictions[mask]
    filtered_true_labels = true_labels[mask]
    f1 = f1_score(filtered_true_labels, filtered_predictions, average='weighted')
    print(f'Test Accuracy for {epoch + 1} is: {accuracy}')
    print(f"Loss for epoch {epoch + 1} is: {epoch_loss/len(train_loader)}")
    print(f'Test F1 Score for {epoch + 1} is: {f1}')

Test Accuracy for 1 is: 0.5793154214206846
Loss for epoch 1 is: 2.2734969902038573
Test F1 Score for 1 is: 0.6743481286432434
Test Accuracy for 2 is: 0.6764814133235186
Loss for epoch 2 is: 1.1988241440057754
Test F1 Score for 2 is: 0.7799248354963891
Test Accuracy for 3 is: 0.6912035333087965
Loss for epoch 3 is: 0.80091158375144
Test F1 Score for 3 is: 0.8010981009934224
Test Accuracy for 4 is: 0.6691203533308796
Loss for epoch 4 is: 0.5146594131737947
Test F1 Score for 4 is: 0.7679069374429932
Test Accuracy for 5 is: 0.6650717703349283
Loss for epoch 5 is: 0.31046628933399917
Test F1 Score for 5 is: 0.7641066924167871
Test Accuracy for 6 is: 0.6886271623113729
Loss for epoch 6 is: 0.17175773289054633
Test F1 Score for 6 is: 0.7933952940767381
Test Accuracy for 7 is: 0.6750092013249908
Loss for epoch 7 is: 0.09949871786870063
Test F1 Score for 7 is: 0.776502114065225
Test Accuracy for 8 is: 0.6820022083179977
Loss for epoch 8 is: 0.047615716536529365
Test F1 Score for 8 is: 0.7841264

# Evaluation

In [33]:
test_data = parse_data('TEST_FILE_FULL.TXT', is_test=False)
test_processed = preprocess_data(test_data, tokenizer, label_map)
input_ids = torch.stack([item['input_ids'] for item in test_processed])
attention_masks = torch.stack([item['attention_mask'] for item in test_processed])
labels_test = torch.tensor([item['label'] for item in test_processed])

test_dataset = TensorDataset(input_ids, attention_masks, labels_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


model.eval()
test_predictions, test_true_labels = [], []
test_sentences = [item['sentence'] for item in test_data]
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs['logits']

        preds = torch.argmax(logits, dim=1)

        test_predictions.extend(preds.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(test_true_labels, test_predictions)
predictions = np.array(test_predictions)
true_labels = np.array(test_true_labels)
mask = true_labels != label_map['Other']
filtered_predictions = predictions[mask]
filtered_true_labels = true_labels[mask]


test_f1 = f1_score(filtered_true_labels, filtered_predictions, average='weighted')

print(f'Validation Accuracy: {test_accuracy}')
print(f'Validation F1 Score: {test_f1}')
# reverse label map
inverse_label_map = {v: k for k, v in label_map.items()}

# print random sample
num_samples = 10
random_indices = random.sample(range(len(test_sentences)), num_samples)
for i in random_indices:
    sentence = test_sentences[i]
    predicted_label = inverse_label_map[test_predictions[i]]
    print(f'{sentence} \n {predicted_label}')

Validation Accuracy: 0.6878910563121089
Validation F1 Score: 0.7934490327549573
We were forced to get off the bus and find accommodation for the night, while the police officers took the bus to the scene of the incident; apparently they were lacking their own transportation. 
 Instrument-Agency(e2,e1)
Optimizing a transformer driven active magnetic shield in induction heating. 
 Component-Whole(e2,e1)
A witch is able to change events by using magic. 
 Instrument-Agency(e2,e1)
It is a tutorial by Mark Levine that aims to summarise the musical theory, including jazz harmony, required by an aspiring jazz pianist. 
 Product-Producer(e2,e1)
The Commission has published an annual report giving a summary of committee activities during the previous year. 
 Message-Topic(e1,e2)
In a cistern that stores cold liquids, the concrete ring serving as juncture element may be thermally lagged against the rock. 
 Other
The world we live in is rooted in an infinite life. 
 Other
Both novels tackle with p